In [35]:
import numpy as np
import pandas as pd
df = pd.read_csv("dirty_cafe_sales.csv")

In [36]:
print(df)

     Transaction ID      Item Quantity Price Per Unit Total Spent  \
0       TXN_1961373    Coffee        2            2.0         4.0   
1       TXN_4977031      Cake        4            3.0        12.0   
2       TXN_4271903    Cookie        4            1.0       ERROR   
3       TXN_7034554     Salad        2            5.0        10.0   
4       TXN_3160411    Coffee        2            2.0         4.0   
...             ...       ...      ...            ...         ...   
9995    TXN_7672686    Coffee        2            2.0         4.0   
9996    TXN_9659401       NaN        3            NaN         3.0   
9997    TXN_5255387    Coffee        4            2.0         8.0   
9998    TXN_7695629    Cookie        3            NaN         3.0   
9999    TXN_6170729  Sandwich        3            4.0        12.0   

      Payment Method  Location Transaction Date  
0        Credit Card  Takeaway       2023-09-08  
1               Cash  In-store       2023-05-16  
2        Credit Card 

In [37]:
print(df.shape)
print(df.dtypes)
print(df.isnull().sum())


(10000, 8)
Transaction ID      str
Item                str
Quantity            str
Price Per Unit      str
Total Spent         str
Payment Method      str
Location            str
Transaction Date    str
dtype: object
Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64


Overview of data types and the amount of incorrect data

In [38]:
df = df.replace(['ERROR', 'UNKNOWN'], pd.NA)
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,NaN,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,NaN,NaN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [39]:
cols = ["Quantity","Price Per Unit", "Total Spent"]
df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')
df["Transaction Date"] = pd.to_datetime(df["Transaction Date"],errors='coerce')
print(df.dtypes)
df.head()

Transaction ID                 str
Item                           str
Quantity                   float64
Price Per Unit             float64
Total Spent                float64
Payment Method                 str
Location                       str
Transaction Date    datetime64[us]
dtype: object


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4.0,1.0,NaN,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2.0,5.0,10.0,NaN,NaN,2023-04-27
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11


Replace datatype from str to float or dateTime


In [40]:
df['Total Spent']=df['Total Spent'].fillna(df['Quantity'] * df['Price Per Unit'])
df.head()


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4.0,1.0,4.0,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2.0,5.0,10.0,NaN,NaN,2023-04-27
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11


Count Total Spent where were NaN, if its didnt helped stay NaN

Was 173 Total Spent with Nan, now 40

In [41]:
df['Quantity']=df['Quantity'].fillna(df['Total Spent'] / df['Price Per Unit'])
df['Price Per Unit']=df['Price Per Unit'].fillna(df['Total Spent'] / df['Quantity'])
df.iloc[15:26]
df[cols].isnull().sum()

Quantity          38
Price Per Unit    38
Total Spent       40
dtype: int64

In [42]:
print(df[['Item', 'Price Per Unit']].dropna().drop_duplicates().sort_values('Price Per Unit'))

        Item  Price Per Unit
2     Cookie             1.0
42       Tea             1.5
0     Coffee             2.0
1       Cake             3.0
17     Juice             3.0
7   Sandwich             4.0
5   Smoothie             4.0
3      Salad             5.0


In [43]:
price_to_item = {
    1.0: 'Cookie',
    1.5: 'Tea',
    2.0: 'Coffee',
    3.0: None,
    4.0: None,
    5.0: 'Salad'
}

df['Item'] = df.apply(
    lambda row: price_to_item.get(row['Price Per Unit'], row['Item'])
    if pd.isna(row['Item']) else row['Item'], axis=1
)

Replace Item name with Price Per Unit if the item has only one unique price

In [44]:
print(df[df['Payment Method'] == "Cash" ])

     Transaction ID      Item  Quantity  Price Per Unit  Total Spent  \
1       TXN_4977031      Cake       4.0             3.0         12.0   
7       TXN_6699534  Sandwich       4.0             4.0         16.0   
10      TXN_2548360     Salad       5.0             5.0         25.0   
12      TXN_7619095  Sandwich       2.0             4.0          8.0   
17      TXN_6769710     Juice       2.0             3.0          6.0   
...             ...       ...       ...             ...          ...   
9987    TXN_1784478     Juice       5.0             3.0         15.0   
9989    TXN_1741685     Juice       5.0             3.0         15.0   
9991    TXN_3897619  Sandwich       3.0             4.0         12.0   
9993    TXN_4766549  Smoothie       2.0             4.0          8.0   
9999    TXN_6170729  Sandwich       3.0             4.0         12.0   

     Payment Method  Location Transaction Date  
1              Cash  In-store       2023-05-16  
7              Cash       NaN       2

In [45]:
print(df["Payment Method"].unique())
print(df["Location"].unique())

<StringArray>
['Credit Card', 'Cash', nan, 'Digital Wallet']
Length: 4, dtype: str
<StringArray>
['Takeaway', 'In-store', nan]
Length: 3, dtype: str


A failed try to determine "Pay metod" by "Location" and vice versa

In [46]:
df.duplicated().sum()

np.int64(0)

Check for duplicates

In [47]:

df.to_csv('cafe_sales_cleaned_test.csv', index=False)

Final import of clean dataset
